In [ ]:
import numpy as np
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from models import *
batch_size = 32
seed = 42
lr = 3e-4
epochs = 15
torch.use_deterministic_algorithms(True, warn_only=True)
device = 'mps'

In [ ]:
def val_evaluate(n, file):
    models = {2: NeuralNet2, 3: NeuralNet3, 4: NeuralNet4, 5: NeuralNet5, 6: NeuralNet6}
    model = models[n]().to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    model.eval()
    loss_function = nn.BCEWithLogitsLoss()
    correct = 0
    total_loss = 0.0
    total = 0
    val_inputs = np.memmap(f'val_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(20000, n, 128, 128, 3))
    val_targets = np.memmap(f'val_data/targets_{n}.dat', dtype=np.uint8, mode='r', shape=(20000,))
    dataset = TensorDataset(torch.from_numpy(val_inputs), torch.from_numpy(val_targets))
    val_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for batch_inputs, batch_targets in val_loader:
            batch_inputs = batch_inputs.float().to(device) / 255.0
            batch_targets = batch_targets.float().to(device)
            inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
            logits = model(*inputs)
            loss = loss_function(logits.squeeze(), batch_targets)
            total_loss += loss.item() * batch_targets.size(0)
            total += batch_targets.size(0)
            predictions = logits.squeeze() > 0
            correct += (predictions == batch_targets).sum().item()
    print("Validation loss:", total_loss / total)
    print("Accuracy:", correct / total)

In [ ]:
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)), 
    transforms.ToTensor()
])

In [ ]:
def train_model(n):
    if not 2 <= n <= 6: raise ValueError("n must be between 2 and 6")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)
    model_classes = {2: NeuralNet2, 3: NeuralNet3, 4: NeuralNet4, 5: NeuralNet5, 6: NeuralNet6}
    model = model_classes[n]().to(device)
    loss_function = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    inputs_mmap = np.memmap(f'train_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(200000, n, 128, 128, 3))
    targets_mmap = np.memmap(f'train_data/targets_{n}.dat', dtype=np.uint8, mode='r', shape=(200000,))
    dataset = TensorDataset(torch.from_numpy(inputs_mmap), torch.from_numpy(targets_mmap))
    generator = torch.Generator()
    generator.manual_seed(seed)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=generator)
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_inputs, batch_targets in train_loader:
            batch_inputs = batch_inputs.float().to(device) / 255.0
            batch_targets = batch_targets.float().to(device)
            inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
            inputs = [torch.stack([train_transforms(img) for img in input]).to(device) for input in inputs]
            optimizer.zero_grad()
            preds = model(*inputs)
            loss = loss_function(preds.squeeze(), batch_targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f'Epoch {epoch+1} avg loss = {running_loss / len(train_loader)}')
        torch.save(model.state_dict(), f'model_{n}/epoch_{epoch+1}.pth')
        val_evaluate(f'model_{n}/epoch_{epoch+1}.pth')

In [ ]:
train_model(2)

Epoch 1 avg loss = 0.59391732006073
Validation loss: 0.4579745085477829
Accuracy: 0.77765
Epoch 2 avg loss = 0.46680512180805206
Validation loss: 0.4160823803424835
Accuracy: 0.80155
Epoch 3 avg loss = 0.430296847679615
Validation loss: 0.389118900346756
Accuracy: 0.81475
Epoch 4 avg loss = 0.40697746134519575
Validation loss: 0.37501611564159393
Accuracy: 0.8254
Epoch 5 avg loss = 0.3905271273136139
Validation loss: 0.3657613921046257
Accuracy: 0.8299
Epoch 6 avg loss = 0.3782141246604919
Validation loss: 0.3531296433210373
Accuracy: 0.8368
Epoch 7 avg loss = 0.3674358264708519
Validation loss: 0.3548219549894333
Accuracy: 0.83545
Epoch 8 avg loss = 0.35927135411739347
Validation loss: 0.33944539716243743
Accuracy: 0.84405
Epoch 9 avg loss = 0.35138597015380857
Validation loss: 0.3488579890489578
Accuracy: 0.8424
Epoch 10 avg loss = 0.3456519784605503
Validation loss: 0.3329165021419525
Accuracy: 0.84865
Epoch 11 avg loss = 0.3387508317148685
Validation loss: 0.3224224383950233
Accura

In [ ]:
train_model(3)

/var/folders/_b/yynhlh9s339c2gvwx12flcqm0000gp/T/ipykernel_21294/3049493972.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  dataset = TensorDataset(torch.from_numpy(inputs_mmap), torch.from_numpy(targets_mmap))


Epoch 1 avg loss = 0.5167346823740006
Validation loss: 0.3954184525489807
Accuracy: 0.8146
Epoch 2 avg loss = 0.4002752219104767
Validation loss: 0.3439425784468651
Accuracy: 0.84295
Epoch 3 avg loss = 0.36391704974651334
Validation loss: 0.3235338572144508
Accuracy: 0.85505
Epoch 4 avg loss = 0.3416021812415123
Validation loss: 0.299694044816494
Accuracy: 0.86875
Epoch 5 avg loss = 0.3266839977109432
Validation loss: 0.29832403988838196
Accuracy: 0.86795
Epoch 6 avg loss = 0.3173937027728558
Validation loss: 0.2867840917110443
Accuracy: 0.87385
Epoch 7 avg loss = 0.30624994275689127
Validation loss: 0.2793344460487366
Accuracy: 0.87755
Epoch 8 avg loss = 0.2977774026441574
Validation loss: 0.27294214494228364
Accuracy: 0.88195
Epoch 9 avg loss = 0.2916968572592735
Validation loss: 0.2720474842071533
Accuracy: 0.8798
Epoch 10 avg loss = 0.28390551075696946
Validation loss: 0.2621239555239677
Accuracy: 0.88615
Epoch 11 avg loss = 0.279449458963871
Validation loss: 0.2642292151689529
Acc

In [ ]:
train_model(4)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1 avg loss = 0.4837610210132599
Validation loss: 0.34611241265535353
Accuracy: 0.8403
Epoch 2 avg loss = 0.36813673577427863
Validation loss: 0.30040791391134264
Accuracy: 0.86675
Epoch 3 avg loss = 0.33186622757196427
Validation loss: 0.27815617768764495
Accuracy: 0.8801
Epoch 4 avg loss = 0.31179343455672265
Validation loss: 0.26651079165935515
Accuracy: 0.88325
Epoch 5 avg loss = 0.29661985114336015
Validation loss: 0.25832597336769103
Accuracy: 0.88795
Epoch 6 avg loss = 0.2849584136772156
Validation loss: 0.2531132567882538
Accuracy: 0.89095
Epoch 7 avg loss = 0.27488975330650806
Validation loss: 0.24761478155851363
Accuracy: 0.89325
Epoch 8 avg loss = 0.26504497078895567
Validation loss: 0.24213221988081932
Accuracy: 0.89535
Epoch 9 avg loss = 0.2590135362672806
Validation loss: 0.24194882246255875
Accuracy: 0.8975
Epoch 10 avg loss = 0.2512863797610998
Validation loss: 0.2302315170943737
Accuracy: 0.9016
Epoch 11 avg loss = 0.24674872449874877
Validation loss: 0.2241059440

In [ ]:
train_model(5)

/var/folders/_b/yynhlh9s339c2gvwx12flcqm0000gp/T/ipykernel_3519/3208410743.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  dataset = TensorDataset(torch.from_numpy(inputs_mmap), torch.from_numpy(targets_mmap))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic su

Epoch 1 avg loss = 0.4551112028384209
Validation loss: 0.33661876776218413
Accuracy: 0.85025
Epoch 2 avg loss = 0.3495742951643467
Validation loss: 0.2860052225232124
Accuracy: 0.872
Epoch 3 avg loss = 0.316156342331171
Validation loss: 0.27103358166217806
Accuracy: 0.8826
Epoch 4 avg loss = 0.297173708230257
Validation loss: 0.2544814642906189
Accuracy: 0.89235
Epoch 5 avg loss = 0.2811030246257782
Validation loss: 0.2495551106095314
Accuracy: 0.89275
Epoch 6 avg loss = 0.26977130842149255
Validation loss: 0.24301367928981782
Accuracy: 0.89635
Epoch 7 avg loss = 0.2608009826630354
Validation loss: 0.24082934670448303
Accuracy: 0.8979
Epoch 8 avg loss = 0.2511209550619125
Validation loss: 0.2445849312722683
Accuracy: 0.8943
Epoch 9 avg loss = 0.24465949345231056
Validation loss: 0.2251642072379589
Accuracy: 0.90395
Epoch 10 avg loss = 0.2373394675463438
Validation loss: 0.22301287412047385
Accuracy: 0.9065
Epoch 11 avg loss = 0.2322888025212288
Validation loss: 0.22664075090885163
Accu

In [ ]:
train_model(6)

/var/folders/_b/yynhlh9s339c2gvwx12flcqm0000gp/T/ipykernel_21097/4034017703.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  dataset = TensorDataset(torch.from_numpy(inputs_mmap), torch.from_numpy(targets_mmap))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic s

Epoch 1 avg loss = 0.4453180294370651
Validation loss: 0.32663458793163297
Accuracy: 0.8566
Epoch 2 avg loss = 0.3388917943513393
Validation loss: 0.27122847965359687
Accuracy: 0.88185
Epoch 3 avg loss = 0.3046012401676178
Validation loss: 0.25295358116626737
Accuracy: 0.89135
Epoch 4 avg loss = 0.2840990759181976
Validation loss: 0.23738557436466218
Accuracy: 0.89965
Epoch 5 avg loss = 0.270021648132205
Validation loss: 0.22994632696509362
Accuracy: 0.90225
Epoch 6 avg loss = 0.25833882723510265
Validation loss: 0.22257117258906364
Accuracy: 0.9074
Epoch 7 avg loss = 0.24849797987282277
Validation loss: 0.22300817896723749
Accuracy: 0.90565
Epoch 8 avg loss = 0.24026132315635682
Validation loss: 0.2160506804049015
Accuracy: 0.90895
Epoch 9 avg loss = 0.23313019252479075
Validation loss: 0.20588333271443843
Accuracy: 0.91505
Epoch 10 avg loss = 0.22444072992682457
Validation loss: 0.20674809060692786
Accuracy: 0.9149
Epoch 11 avg loss = 0.21851453195124865
Validation loss: 0.2067332242